In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

import sys
sys.path.append("../../utils/")

from utils import *

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_EXPERIMENTO = "CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1"
CARPETA_DATASET = "CIC17__NearMiss_SMOTE_ENN__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== CONFIG KNN =====
N_NEIGHBORS = 5
WEIGHTS = "distance"      # "uniform" o "distance"
METRIC = "minkowski"      # euclidean suele ser minkowski con p=2
P = 2

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 4

In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/CIC17__NearMiss_SMOTE_ENN__v1/CIC17__NearMiss_SMOTE_ENN__v1__train.csv

Ruta dataset test:
/home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/CIC17__NearMiss_SMOTE_ENN__v1/CIC17__NearMiss_SMOTE_ENN__v1__test.csv

Ruta resultados:
/home/javier/TFG_MODELOS_SIMPLES2/04_experimentos/logs/resultados/CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(140631, 48)


,DESTINATION_PORT,FLOW_DURATION,TOTAL_FWD_PACKETS,TOTAL_LENGTH_OF_FWD_PACKETS,FWD_PACKET_LENGTH_MAX,FWD_PACKET_LENGTH_MIN,FWD_PACKET_LENGTH_MEAN,BWD_PACKET_LENGTH_MAX,BWD_PACKET_LENGTH_MIN,FLOW_BYTES_S,...,INIT_WIN_BYTES_FORWARD,INIT_WIN_BYTES_BACKWARD,ACT_DATA_PKT_FWD,MIN_SEG_SIZE_FORWARD,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_STD,LABEL
0,443,119985408,257,224020,14628,0,871.673152,1075,0,2373.255255,...,258,2043,124,20,0.0,0.0,0,0,0.0,0
1,443,118709021,3141,111133,944,0,35.381407,4549,2,71774.899060,...,3165,348,118,20,0.0,0.0,0,0,0.0,0
2,443,118451558,18605,156557,486,0,8.414781,4380,6,471911.437400,...,7294,437,18591,20,0.0,0.0,0,0,0.0,0
3,443,118190125,2743,113004,944,0,41.197229,4484,0,62152.561390,...,3758,348,120,20,0.0,0.0,0,0,0.0,0
4,443,118034742,18494,155286,486,0,8.396561,8760,6,470791.430200,...,3798,437,18302,20,0.0,0.0,0,0,0.0,0


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
0,10000
14,10000
2,9995
9,9977
1,9968
3,9960
4,9959
12,9935
5,9923


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (140631, 47)
Shape y_train: (140631,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
pipeline = Pipeline([
    ("scaler", RobustScaler()),
    ("pca", PCA(n_components=N_COMPONENTS_PCA)),
    ("knn", KNeighborsClassifier(
        n_neighbors=N_NEIGHBORS,
        weights=WEIGHTS,
        metric=METRIC,
        p=P
    ))
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"with_centering with_centering: bool, default=TrueIf `True`, center the data before scaling.This will cause :meth:`transform` to raise an exception when attemptedon sparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_scaling with_scaling: bool, default=TrueIf `True`, scale the data to interquartile range.",True
,"quantile_range quantile_range: tuple (q_min, q_max), 0.0 < q_min < q_max < 100.0, default=(25.0, 75.0)Quantile range used to calculate `scale_`. By default this is equal tothe IQR, i.e., `q_min` is the first quantile and `q_max` is the thirdquantile... versionadded:: 0.18","(25.0, ...)"
,"copy copy: bool, default=TrueIf `False`, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"unit_variance unit_variance: bool, default=FalseIf `True`, scale data so that normally distributed features have avariance of 1. In general, if the difference between the x-values of`q_max` and `q_min` for a standard normal distribution is greaterthan 1, the dataset will be scaled down. If less than 1, the datasetwill be scaled up... versionadded:: 0.24",False
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",4
,"copy copy: bool, default=TrueIf False, data passed to fit are 

In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [10]:
scoring = {
    "accuracy": "accuracy",

    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",

    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",

    "mcc": make_scorer(matthews_corrcoef),

    "roc_auc": "roc_auc_ovr_weighted",
}

In [11]:
cv_results = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

cv_results.keys()

dict_keys(['fit_time', 'score_time', 'test_accuracy', 'test_precision_weighted', 'test_recall_weighted', 'test_f1_weighted', 'test_precision_macro', 'test_recall_macro', 'test_f1_macro', 'test_mcc', 'test_roc_auc'])

In [12]:
df_folds = pd.DataFrame({
    "fold": np.arange(1, N_SPLITS + 1),

    "accuracy": cv_results["test_accuracy"],

    "precision_weighted": cv_results["test_precision_weighted"],
    "recall_weighted": cv_results["test_recall_weighted"],
    "f1_weighted": cv_results["test_f1_weighted"],

    "precision_macro": cv_results["test_precision_macro"],
    "recall_macro": cv_results["test_recall_macro"],
    "f1_macro": cv_results["test_f1_macro"],

    "mcc": cv_results["test_mcc"],

    "roc_auc": cv_results["test_roc_auc"],

    "fit_time": cv_results["fit_time"],
    "score_time": cv_results["score_time"]
})

df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,0.903936,0.903765,0.903936,0.903772,0.889369,0.889196,0.889198,0.896929,0.980169,0.158871,0.210651
1,2,0.906350,0.906188,0.906350,0.906136,0.891669,0.891355,0.891380,0.899527,0.981139,0.206089,0.177603
2,3,0.902083,0.901900,0.902083,0.901788,0.888342,0.888188,0.888060,0.894961,0.979958,0.217358,0.179529
3,4,0.901124,0.900639,0.901124,0.900710,0.885228,0.885168,0.885016,0.893924,0.979376,0.175677,0.189479
4,5,0.901657,0.901584,0.901657,0.901432,0.886898,0.886750,0.886642,0.894502,0.979464,0.167223,0.176789


In [13]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_train": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_train": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "n_neighbors": N_NEIGHBORS,
        "weights": WEIGHTS,
        "metric": METRIC,
        "p": P,
        "n_components_pca": N_COMPONENTS_PCA
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "roc_auc": float(df_folds["roc_auc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "roc_auc": float(df_folds["roc_auc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

{'experimento': 'CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1',
 'dataset_train': '/home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/CIC17__NearMiss_SMOTE_ENN__v1/CIC17__NearMiss_SMOTE_ENN__v1__train.csv',
 'shape_train': {'rows': 140631, 'cols': 48},
 'parametros': {'label_col': 'LABEL',
  'n_splits': 5,
  'shuffle': True,
  'random_state': 42,
  'n_neighbors': 5,
  'weights': 'distance',
  'metric': 'minkowski',
  'p': 2,
  'n_components_pca': 4},
 'metricas_media': {'accuracy': 0.9030299087270132,
  'precision_weighted': 0.9028150142461977,
  'recall_weighted': 0.9030299087270132,
  'f1_weighted': 0.9027677141118579,
  'precision_macro': 0.8883010816270289,
  'recall_macro': 0.8881314920206366,
  'f1_macro': 0.8880592659882887,
  'mcc': 0.8959684497303894,
  'roc_auc': 0.9800212871014379,
  'fit_time': 0.1850438117980957,
  'score_time': 0.18681039810180664},
 'metricas_std': {'accuracy': 0.0021365206092529604,
  'precision_weighted': 0.002200246681318255,
  'recall_weighted': 

In [14]:
print("========== RESULTADOS CV ==========")
print(f"Accuracy            : {summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}")
print()

print(f"Precision weighted  : {summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}")
print(f"Recall weighted     : {summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}")
print(f"F1 weighted         : {summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}")
print(f"Recall macro        : {summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}")
print(f"F1 macro            : {summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}")
print()

print(f"MCC                 : {summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}")
print(f"ROC AUC              : {summary_cv['metricas_media']['roc_auc']:.6f} ± {summary_cv['metricas_std']['roc_auc']:.6f}")
print()
print(f"Fit time medio      : {summary_cv['metricas_media']['fit_time']:.6f}")
print(f"Score time medio    : {summary_cv['metricas_media']['score_time']:.6f}")

========== RESULTADOS CV ==========
Accuracy            : 0.903030 ± 0.002137

Precision weighted  : 0.902815 ± 0.002200
Recall weighted     : 0.903030 ± 0.002137
F1 weighted         : 0.902768 ± 0.002198

Precision macro     : 0.888301 ± 0.002444
Recall macro        : 0.888131 ± 0.002357
F1 macro            : 0.888059 ± 0.002429

MCC                 : 0.895968 ± 0.002288
ROC AUC              : 0.980021 ± 0.000707

Fit time medio      : 0.185044
Score time medio    : 0.186810


In [15]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

Resultados por fold guardados en:
/home/javier/TFG_MODELOS_SIMPLES2/04_experimentos/logs/resultados/CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1/CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1__folds.csv


In [16]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

Resumen CV guardado en:
/home/javier/TFG_MODELOS_SIMPLES2/04_experimentos/logs/resultados/CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1/CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1__summary_cv.json


In [17]:
df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,0.903936,0.903765,0.903936,0.903772,0.889369,0.889196,0.889198,0.896929,0.980169,0.158871,0.210651
1,2,0.906350,0.906188,0.906350,0.906136,0.891669,0.891355,0.891380,0.899527,0.981139,0.206089,0.177603
2,3,0.902083,0.901900,0.902083,0.901788,0.888342,0.888188,0.888060,0.894961,0.979958,0.217358,0.179529
3,4,0.901124,0.900639,0.901124,0.900710,0.885228,0.885168,0.885016,0.893924,0.979376,0.175677,0.189479
4,5,0.901657,0.901584,0.901657,0.901432,0.886898,0.886750,0.886642,0.894502,0.979464,0.167223,0.176789


In [18]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

Forma del dataset test:
(504160, 48)


,DESTINATION_PORT,FLOW_DURATION,TOTAL_FWD_PACKETS,TOTAL_LENGTH_OF_FWD_PACKETS,FWD_PACKET_LENGTH_MAX,FWD_PACKET_LENGTH_MIN,FWD_PACKET_LENGTH_MEAN,BWD_PACKET_LENGTH_MAX,BWD_PACKET_LENGTH_MIN,FLOW_BYTES_S,...,INIT_WIN_BYTES_FORWARD,INIT_WIN_BYTES_BACKWARD,ACT_DATA_PKT_FWD,MIN_SEG_SIZE_FORWARD,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_STD,LABEL
0,443,93606787,18,3128,410,6,173.777778,1618,38,186.845426,...,256,8192,17,20,32068.8000,1749.725464,34107,30651,6.931172e+06,0
1,80,98331732,6,354,336,0,59.000000,4344,0,121.517233,...,0,235,3,20,21035.0000,0.000000,21035,21035,0.000000e+00,1
2,80,117175775,231,1427,401,0,6.177489,4584,0,5590.916723,...,29200,1039,4,32,206922.5455,636207.069100,2125159,14988,8.932177e+04,0
3,80,1285910,3,566,560,0,188.666667,1894,2,1923.929357,...,29200,15680,2,20,0.0000,0.000000,0,0,0.000000e+00,0
4,53,47941,1,44,44,44,44.000000,224,224,5590.204627,...,-1,-1,0,40,0.0000,0.000000,0,0,0.000000e+00,0


In [19]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna test: LABEL
Tipo de LABEL test: int64

Distribución de clases en test:


,count
LABEL,
0,419012
1,34569
2,25603
3,18139
4,2057
5,1186
6,1077
7,1046
8,644


In [20]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

Shape X_test: (504160, 47)
Shape y_test: (504160,)


In [21]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

Columnas no numéricas en X_test:
[]


In [22]:
pipeline.fit(X_train, y_train)

print("Modelo final entrenado con todo el dataset train.")

Modelo final entrenado con todo el dataset train.


In [23]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

y_proba_test = pipeline.predict_proba(X_test)

roc_auc_test = roc_auc_score(
    y_test,
    y_proba_test,
    multi_class="ovr",
    average="weighted"
)

Predicciones en test generadas.
Número de predicciones: 504160


In [24]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "roc_auc": roc_auc_test,

    "mcc": matthews_corrcoef(y_test, y_pred_test)
}

metricas_test

{'accuracy': 0.15149158997143763,
 'precision_weighted': 0.8694885198276697,
 'recall_weighted': 0.15149158997143763,
 'f1_weighted': 0.18732236481252046,
 'precision_macro': 0.12891989545898236,
 'recall_macro': 0.600296927375855,
 'f1_macro': 0.09833482504719147,
 'roc_auc': 0.5948631883125636,
 'mcc': 0.14349866423204033}

In [25]:
print("========== RESULTADOS TEST ==========")
print(f"Accuracy            : {metricas_test['accuracy']:.6f}")
print()

print(f"Precision weighted  : {metricas_test['precision_weighted']:.6f}")
print(f"Recall weighted     : {metricas_test['recall_weighted']:.6f}")
print(f"F1 weighted         : {metricas_test['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {metricas_test['precision_macro']:.6f}")
print(f"Recall macro        : {metricas_test['recall_macro']:.6f}")
print(f"F1 macro            : {metricas_test['f1_macro']:.6f}")
print()

print(f"MCC                 : {metricas_test['mcc']:.6f}")

print(f"ROC AUC              : {metricas_test['roc_auc']:.6f}")

========== RESULTADOS TEST ==========
Accuracy            : 0.151492

Precision weighted  : 0.869489
Recall weighted     : 0.151492
F1 weighted         : 0.187322

Precision macro     : 0.128920
Recall macro        : 0.600297
F1 macro            : 0.098335

MCC                 : 0.143499
ROC AUC              : 0.594863


In [26]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

Matriz de confusión en test:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,37518,36154,25260,26273,15198,70186,14433,23522,26002,44665,1434,3109,84225,10048,985
1,92,19734,1532,387,8127,1122,707,147,862,538,286,569,56,340,70
2,15,3626,10580,37,2856,3110,1405,210,1202,399,36,67,1728,332,0
3,1,38,12,2648,4,13,1832,5,1071,6728,3639,10,1,2137,0
4,0,117,22,2,1679,12,31,12,42,33,11,4,8,84,0
5,0,8,4,9,11,1115,2,2,21,5,3,0,2,4,0
6,0,0,1,15,31,1,977,17,1,10,9,1,2,11,1
7,0,2,0,1,4,0,9,1016,3,6,2,0,1,2,0
8,1,14,0,19,12,25,5,0,549,7,9,0,0,3,0
9,0,1,0,8,14,4,2,0,1,344,5,0,2,9,0


In [27]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

========== CLASSIFICATION REPORT TEST ==========
              precision    recall  f1-score   support

           0       1.00      0.09      0.16    419012
           1       0.33      0.57      0.42     34569
           2       0.28      0.41      0.34     25603
           3       0.09      0.15      0.11     18139
           4       0.06      0.82      0.11      2057
           5       0.01      0.94      0.03      1186
           6       0.05      0.91      0.10      1077
           7       0.04      0.97      0.08      1046
           8       0.02      0.85      0.04       644
           9       0.01      0.88      0.01       390
          10       0.03      0.52      0.05       294
          11       0.01      0.43      0.03       130
          12       0.00      0.71      0.00         7
          13       0.00      0.25      0.00         4
          14       0.00      0.50      0.00         2

    accuracy                           0.15    504160
   macro avg       0.13      0.

In [28]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_neighbors": N_NEIGHBORS,
        "weights": WEIGHTS,
        "metric": METRIC,
        "p": P,
        "n_components_pca": N_COMPONENTS_PCA
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "mcc": float(metricas_test["mcc"])
    }
}

summary_test

{'experimento': 'CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1',
 'dataset_test': '/home/javier/TFG_MODELOS_SIMPLES2/02_datasets/processed/CIC17__NearMiss_SMOTE_ENN__v1/CIC17__NearMiss_SMOTE_ENN__v1__test.csv',
 'shape_test': {'rows': 504160, 'cols': 48},
 'parametros': {'label_col': 'LABEL',
  'n_neighbors': 5,
  'weights': 'distance',
  'metric': 'minkowski',
  'p': 2,
  'n_components_pca': 4},
 'metricas_test': {'accuracy': 0.15149158997143763,
  'precision_weighted': 0.8694885198276697,
  'recall_weighted': 0.15149158997143763,
  'f1_weighted': 0.18732236481252046,
  'precision_macro': 0.12891989545898236,
  'recall_macro': 0.600296927375855,
  'f1_macro': 0.09833482504719147,
  'mcc': 0.14349866423204033}}

In [29]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

Métricas test guardadas en:
/home/javier/TFG_MODELOS_SIMPLES2/04_experimentos/logs/resultados/CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1/CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1__metricas_test.csv


In [30]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

Matriz de confusión test guardada en:
/home/javier/TFG_MODELOS_SIMPLES2/04_experimentos/logs/resultados/CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1/CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1__confusion_matrix_test.csv


In [31]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

Resumen test guardado en:
/home/javier/TFG_MODELOS_SIMPLES2/04_experimentos/logs/resultados/CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1/CIC17__NearMiss_SMOTE_ENN__v1__pca4_knn__v1__summary_test.json


In [32]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])

========== RESUMEN FINAL ==========
CV:
{'accuracy': 0.9030299087270132, 'precision_weighted': 0.9028150142461977, 'recall_weighted': 0.9030299087270132, 'f1_weighted': 0.9027677141118579, 'precision_macro': 0.8883010816270289, 'recall_macro': 0.8881314920206366, 'f1_macro': 0.8880592659882887, 'mcc': 0.8959684497303894, 'roc_auc': 0.9800212871014379, 'fit_time': 0.1850438117980957, 'score_time': 0.18681039810180664}

TEST:
{'accuracy': 0.15149158997143763, 'precision_weighted': 0.8694885198276697, 'recall_weighted': 0.15149158997143763, 'f1_weighted': 0.18732236481252046, 'precision_macro': 0.12891989545898236, 'recall_macro': 0.600296927375855, 'f1_macro': 0.09833482504719147, 'mcc': 0.14349866423204033}
